In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("26-explain-deep-dive")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
dept = dfs["departments"]

## Section 1 — explain() modes

In [ ]:
print("=" * 70)
print("SECTION 1: explain() Modes")
print("=" * 70)

# extended=False (default): physical plan only
print("\n--- default: physical plan only ---")
emp.filter(F.col("dept_id") == 1).select("emp_id", "salary").explain()

print("\n--- extended=True (parsed/analyzed/optimized/physical plans) ---")
emp.filter(F.col("dept_id") == 1).select("emp_id", "salary").explain(extended=True)

print("\n--- mode='simple' ---")
emp.filter(F.col("dept_id") == 1).explain(mode="simple")

print("\n--- mode='extended' ---")
emp.filter(F.col("dept_id") == 1).explain(mode="extended")

print("\n--- mode='codegen' ---")
emp.filter(F.col("dept_id") == 1).explain(mode="codegen")

## Section 2 — Reading the physical plan: key nodes

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: Physical Plan Node Reference (comments)")
print("=" * 70)

# Nodes to recognize:
# FileScan / InMemoryTableScan — how data is read
# Filter                       — predicate; "pushed" means applied at read time
# Project                      — column selection
# Exchange                     — shuffle (wide transformation)
# Sort                         — often precedes SortMergeJoin
# HashAggregate                — groupBy aggregation (partial + final)
# BroadcastHashJoin            — fast join using broadcast
# SortMergeJoin                — standard join with shuffle
# BroadcastNestedLoopJoin      — non-equi or small × small join
print("(See inline comments above for node descriptions)")

## Section 3 — Predicate pushdown in plan

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Predicate Pushdown — filter placement in plan")
print("=" * 70)

from pyspark.sql.functions import broadcast

# Show: filter after join (Catalyst may still push it down — verify in plan)
print("\n[Filter AFTER join] — does Catalyst push the filter down?")
emp.join(broadcast(dept), "dept_id", "left") \
   .filter(F.col("dept_id") == 1) \
   .explain()

# vs filter before join: explicit pushdown
print("\n[Filter BEFORE join] — explicit pushdown:")
emp.filter(F.col("dept_id") == 1) \
   .join(broadcast(dept), "dept_id", "left") \
   .explain()

## Section 4 — Exchange (shuffle) node detection

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Exchange (Shuffle) Node Detection")
print("=" * 70)

# groupBy triggers a shuffle → look for Exchange in plan
print("\n[groupBy] — one Exchange expected:")
emp.groupBy("dept_id").agg(F.avg("salary")).explain()

# join without broadcast → two Exchanges (one per side)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("\n[join, no broadcast] — two Exchange nodes expected (SortMergeJoin):")
emp.join(sal, "emp_id").explain()  # should show Exchange nodes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

## Section 5 — Whole-stage code generation (WholeStageCodegen)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: WholeStageCodegen (fused operators)")
print("=" * 70)

# Lines prefixed with "*" in physical plan are WholeStageCodegen stages.
# Multiple operators are fused into one JVM bytecode function for efficiency.
print("\n[filter + withColumn + select] — look for *(1) prefix on fused operators:")
emp.filter(F.col("salary") > 100000) \
   .withColumn("bonus", F.col("salary") * 0.1) \
   .select("emp_id", "salary", "bonus") \
   .explain()
# *(1) Project and *(1) Filter in the same codegen stage = fused into one pass

## Section 6 — Spark SQL EXPLAIN

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Spark SQL EXPLAIN statements")
print("=" * 70)

print("\n[EXPLAIN via SQL]:")
spark.sql("EXPLAIN SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id").show(truncate=False)

print("\n[EXPLAIN EXTENDED via SQL]:")
spark.sql("EXPLAIN EXTENDED SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id").show(truncate=False)

## Section 7 — Reading window function plan

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Window Function Plan")
print("=" * 70)

w = Window.partitionBy("emp_id").orderBy("effective_date")
print("\n[row_number() window] — look for: Window, Sort, Exchange nodes:")
sal.withColumn("rn", F.row_number().over(w)).explain()
# Expected nodes: Window [row_number() ...], Sort [emp_id, effective_date ASC], Exchange

## Section 8 — .explain() on write plan

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: explain() Logical Plan Before Write")
print("=" * 70)

out = output_path("parquet/emp_explain_test")
writer = emp.filter(F.col("dept_id") == 1).write.mode("overwrite")

# Explain the logical plan before writing
print("\n[Plan for the filtered DF that will be written]:")
emp.filter(F.col("dept_id") == 1).explain()

# Trigger the write
writer.parquet(out)
print("Written to:", out)

stop_and_wait(spark)